# Assignment 1: Mini Data Pipeline

This notebook is a simple, commented solution.

Place `events.csv` in the same folder as this notebook, then run cells.

In [2]:
import pandas as pd

# Show useful context when an exception occurs.
%xmode Context

Exception reporting mode: Context


## 1. Load and clean the data

In [3]:
# Read every column without forcing a type yet.
# The cleaning functions below will perform safe conversions.
raw_df = pd.read_csv("events.csv")
raw_df.head()

,event_id,event_timestamp,customer_id,product_id,event_type,quantity,unit_price,country,source,status
0,E000157,2026-08-13 13:06:46,C00059,P0027,purchase,2,110.37,MX,mobile,processed
1,E000021,2026-08-13 21:51:05,C00125,P0053,view,1,25.99,US,partner_api,ok
2,E000356,2026-08-08 22:22:55,C00113,P0056,purchase,1,205.10,US,web,ok
3,E000006,2026-08-01 23:43:46,C00032,P0060,view,1,117.56,GB,web,failed
4,E000124,2026-08-12 02:29:55,C00018,P0016,purchase,5,13.80,MX,web,failed


In [4]:
raw_df.columns

Index(['event_id', 'event_timestamp', 'customer_id', 'product_id',
       'event_type', 'quantity', 'unit_price', 'country', 'source', 'status'],
      dtype='str')

In [5]:
raw_df.drop?

Signature:
raw_df.drop(
    labels: 'IndexLabel | ListLike' = None,
    *,
    axis: 'Axis' = 0,
    index: 'IndexLabel | ListLike' = None,
    columns: 'IndexLabel | ListLike' = None,
    level: 'Level | None' = None,
    inplace: 'bool' = False,
    errors: 'IgnoreRaise' = 'raise',
) -> 'DataFrame | None'
Docstring:
Drop specified labels from rows or columns.

Remove rows or columns by specifying label names and corresponding
axis, or by directly specifying index or column names. When using a
multi-index, labels on different levels can be removed by specifying
the level. See the :ref:`user guide <advanced.shown_levels>`
for more information about the now unused levels.

Parameters
----------
labels : single label or iterable of labels
    Index or column labels to drop. A tuple will be used as a single
    label and not treated as an iterable.
axis : {0 or 'index', 1 or 'columns'}, default 0
    Whether to drop labels from the index (0 or 'index') or
    columns (1 or 'columns').
ind

In [6]:
help(raw_df.groupby)

Help on method groupby in module pandas.core.frame:

groupby(
    by=None,
    level: IndexLabel | None = None,
    *,
    as_index: bool = True,
    sort: bool = True,
    group_keys: bool = True,
    observed: bool = True,
    dropna: bool = True
) -> DataFrameGroupBy method of pandas.DataFrame instance
    Group DataFrame using a mapper or by a Series of columns.

    A groupby operation involves some combination of splitting the
    object, applying a function, and combining the results. This can be
    used to group large amounts of data and compute operations on these
    groups.

    Parameters
    ----------
    by : mapping, function, label, pd.Grouper or list of such
        Used to determine the groups for the groupby.
        If ``by`` is a function, it's called on each value of the object's
        index. If a dict or Series is passed, the Series or dict VALUES
        will be used to determine the groups (the Series' values are first
        aligned; see ``.align()`` meth

In [7]:
def standardize_text(df):
    """Trim whitespace and convert selected categories to lowercase."""
    result = df.copy()
    for column in ["event_type", "status"]:
        result[column] = result[column].astype("string").str.strip().str.lower()
    return result


def convert_types(df):
    """Convert dates and numbers; invalid values become missing (NaT or NaN)."""
    result = df.copy()
    result["event_timestamp"] = pd.to_datetime(
        result["event_timestamp"], errors="coerce"
    )
    result["quantity"] = pd.to_numeric(result["quantity"], errors="coerce")
    result["unit_price"] = pd.to_numeric(result["unit_price"], errors="coerce")
    

    # Treat non-positive quantities and negative prices as invalid.
    result["quantity"] = result["quantity"].mask(result["quantity"] <= 0)
    result["unit_price"] = result["unit_price"].mask(result["unit_price"] < 0)
    return result


def remove_duplicates(df):
    """Remove rows that are exact duplicates."""
    return df.drop_duplicates().copy()

In [17]:
# Call the three cleaning functions in a clear pipeline.
clean_df = standardize_text(raw_df)
clean_df = convert_types(clean_df)
clean_df = remove_duplicates(clean_df)

# Identify records containing important missing or invalid values.
invalid_mask = (
    clean_df["event_timestamp"].isna()
    | clean_df["quantity"].isna()
    | clean_df["unit_price"].isna()
    | clean_df["customer_id"].isna()
    | clean_df["product_id"].isna()
)

print("Rows containing at least one important missing/invalid value:", invalid_mask.sum())
clean_df.head()

Rows containing at least one important missing/invalid value: 49


,event_id,event_timestamp,customer_id,product_id,event_type,quantity,unit_price,country,source,status
0,E000157,2026-08-13 13:06:46,C00059,P0027,purchase,2.0,110.37,MX,mobile,processed
1,E000021,2026-08-13 21:51:05,C00125,P0053,view,1.0,25.99,US,partner_api,ok
2,E000356,2026-08-08 22:22:55,C00113,P0056,purchase,1.0,205.10,US,web,ok
3,E000006,2026-08-01 23:43:46,C00032,P0060,view,1.0,117.56,GB,web,failed
4,E000124,2026-08-12 02:29:55,C00018,P0016,purchase,5.0,13.80,MX,web,failed


In [18]:
df = clean_df[~invalid_mask]
df.head()

,event_id,event_timestamp,customer_id,product_id,event_type,quantity,unit_price,country,source,status
0,E000157,2026-08-13 13:06:46,C00059,P0027,purchase,2.0,110.37,MX,mobile,processed
1,E000021,2026-08-13 21:51:05,C00125,P0053,view,1.0,25.99,US,partner_api,ok
2,E000356,2026-08-08 22:22:55,C00113,P0056,purchase,1.0,205.10,US,web,ok
3,E000006,2026-08-01 23:43:46,C00032,P0060,view,1.0,117.56,GB,web,failed
4,E000124,2026-08-12 02:29:55,C00018,P0016,purchase,5.0,13.80,MX,web,failed


In [20]:
len(df)

371

In [22]:
len(clean_df)

420

## 2. Debug the broken transformation

The original function has several problems. It contains spaces inside column names such as `" quantity "`, uses inconsistent text (`" Purchase "`) after the data has been standardized to lowercase, and misspells `revenue` as `reveneu`. The traceback points to the first invalid column lookup, which helps locate the problem.

In [9]:
# Intentionally broken version from the assignment.
# Uncomment the last line to produce and inspect its traceback.
def calculate_revenue_broken(df):
    df[" revenue "] = df[" quantity "] * df[" unit_price "]
    purchases = df[df[" event_type "] == " Purchase "]
    return purchases.groupby(" country ")[" reveneu "].sum()

# calculate_revenue_broken(clean_df)

In [10]:
calculate_revenue_broken(clean_df)

KeyError: ' quantity '

In [11]:
def calculate_revenue(df):
    """Return valid purchase revenue grouped by country."""
    result = df.copy()

    # Revenue is missing if quantity or unit price is missing.
    result["revenue"] = result["quantity"] * result["unit_price"]

    # Keep purchases with all values needed for the calculation.
    purchases = result.loc[
        result["event_type"].eq("purchase")
        & result["country"].notna()
        & result["revenue"].notna()
    ]

    return purchases.groupby("country")["revenue"].sum().sort_index()


purchase_revenue = calculate_revenue(clean_df)
purchase_revenue

country
CA    6654.89
DE    9732.48
GB    8692.54
MX    6955.25
US    3704.78
Name: revenue, dtype: float64

**Debugging note:** With `%xmode Context`, the traceback shows the failing line and nearby code. The `KeyError` reveals that a requested column name does not exist, which leads us to remove the extra spaces and correct the misspelling.

## 3. Compare loop and vectorized implementations

In [35]:
help(pd.testing)

Help on module pandas.testing in pandas:

NAME
    pandas.testing - Public testing utility functions.

FUNCTIONS
    assert_extension_array_equal(
        left,
        right,
        check_dtype: bool | Literal['equiv'] = True,
        index_values=None,
        check_exact: bool | lib.NoDefault = <no_default>,
        rtol: float | lib.NoDefault = <no_default>,
        atol: float | lib.NoDefault = <no_default>,
        obj: str = 'ExtensionArray'
    ) -> None
        Check that left and right ExtensionArrays are equal.

        This method compares two ``ExtensionArray`` instances for equality,
        including checks for missing values, the dtype of the arrays, and
        the exactness of the comparison (or tolerance when comparing floats).

        Parameters
        ----------
        left, right : ExtensionArray
            The two arrays to compare.
        check_dtype : bool, default True
            Whether to check if the ExtensionArray dtypes are identical.
        index

In [36]:
def revenue_by_country_loop(df):
    """Version A: calculate purchase revenue one row at a time."""
    totals = {}

    for row in df.itertuples(index=False):
        if (
            row.event_type == "purchase"
            and pd.notna(row.country)
            and pd.notna(row.quantity)
            and pd.notna(row.unit_price)
        ):
            revenue = row.quantity * row.unit_price
            totals[row.country] = totals.get(row.country, 0) + revenue

    return pd.Series(totals, dtype=float).sort_index()


def revenue_by_country_vectorized(df):
    """Version B: use pandas filtering, arithmetic, and groupby."""
    purchases = df.loc[df["event_type"].eq("purchase")].copy()
    purchases["revenue"] = purchases["quantity"] * purchases["unit_price"]
    return purchases.groupby("country")["revenue"].sum().sort_index()

In [37]:
# Confirm that both implementations produce the same result.
pd.testing.assert_series_equal(
    revenue_by_country_loop(clean_df),
    revenue_by_country_vectorized(clean_df),
    check_names=False,
)
print("Both versions produce the same result.")

Both versions produce the same result.


In [27]:
# Run each function repeatedly for a more reliable timing comparison.
%timeit revenue_by_country_loop(clean_df)
%timeit revenue_by_country_vectorized(clean_df)

676 μs ± 9.72 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
302 μs ± 4.09 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [28]:
# Profile the vectorized implementation once.
%prun revenue_by_country_vectorized(clean_df)

         2889 function calls (2839 primitive calls) in 0.001 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.001    0.001 926903926.py:18(revenue_by_country_vectorized)
  581/569    0.000    0.000    0.000    0.000 {built-in method builtins.isinstance}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
       15    0.000    0.000    0.000    0.000 generic.py:6119(__finalize__)
       10    0.000    0.000    0.000    0.000 take.py:118(_take_nd_ndarray)
       73    0.000    0.000    0.000    0.000 __init__.py:330(_compile)
    18/10    0.000    0.000    0.000    0.000 take.py:57(take_nd)
        6    0.000    0.000    0.000    0.000 managers.py:1226(iget)
       73    0.000    0.000    0.000    0.000 __init__.py:174(search)
       12    0.000    0.000    0.000    0.000 cast.py:451(maybe_promote)
        2    0.000    0.000    0.000    0.000 series.p

**Timing and profiling interpretation:** Record the actual result shown above. For sufficiently large datasets, the vectorized pandas version is normally faster because it avoids a Python-level loop over rows. The profiler shows where time is spent, such as filtering, multiplication, and grouping. Profiling matters because it directs optimization effort toward real pipeline bottlenecks instead of guesses.

## 4. Pipeline quality summary

In [29]:
def pipeline_summary(raw_df, clean_df):
    """Return the required pipeline data-quality metrics."""
    purchase_revenue = calculate_revenue(clean_df)

    summary = {
        "raw_rows": len(raw_df),
        "cleaned_rows": len(clean_df),
        "duplicate_rows_removed": len(raw_df) - len(clean_df),
        "unusable_timestamps": int(clean_df["event_timestamp"].isna().sum()),
        "missing_customer_or_product_ids": int(
            (clean_df["customer_id"].isna() | clean_df["product_id"].isna()).sum()
        ),
        "total_valid_purchase_revenue": float(purchase_revenue.sum()),
    }

    return pd.Series(summary, name="value")


summary = pipeline_summary(raw_df, clean_df)
summary

raw_rows                             432.00
cleaned_rows                         420.00
duplicate_rows_removed                12.00
unusable_timestamps                   10.00
missing_customer_or_product_ids       23.00
total_valid_purchase_revenue       35739.94
Name: value, dtype: float64

## Production decision

I would not automatically load every record into the production analytics table. Records with unusable timestamps, missing identifiers, or invalid revenue inputs should first be quarantined for review, while valid records may continue through the pipeline. I would allow the valid subset to proceed only after confirming that duplicate removal and missing-value counts are within agreed data-quality thresholds.